In [5]:
# ===========================
# STEP 2 : Migration des données vers MongoDB
# ===========================

# ===========================
# CELLULE 1 : Import des librairies
# ===========================
import boto3
import pandas as pd
import json
import os
from pymongo import MongoClient
import certifi


print("✅ Librairies importées")

# ===========================
# CELLULE 2 : Configuration S3
# ===========================
BUCKET_NAME = "greencoop-forecast2-raw"
REGION_NAME = "eu-west-3"

AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")

print(f"AWS_ACCESS_KEY_ID chargé : {AWS_ACCESS_KEY_ID is not None}")
print(f"AWS_SECRET_ACCESS_KEY chargé : {AWS_SECRET_ACCESS_KEY is not None}")

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    region_name=REGION_NAME
)
print(f"✅ Connexion S3 établie pour le bucket {BUCKET_NAME}")

# ===========================
# CELLULE 3 : Fonctions pour lire les fichiers
# ===========================
def read_jsonl_from_s3(s3_path):
    response = s3.get_object(Bucket=BUCKET_NAME, Key=s3_path)
    data = response['Body'].read().decode('utf-8').splitlines()
    records = [json.loads(line)['_airbyte_data'] for line in data]
    df = pd.DataFrame(records)
    print(f"Lecture {s3_path} : {len(df)} lignes, {len(df.columns)} colonnes")
    return df

def read_excel_from_s3(s3_path):
    response = s3.get_object(Bucket=BUCKET_NAME, Key=s3_path)
    df = pd.read_excel(response['Body'])
    print(f"Lecture {s3_path} : {len(df)} lignes, {len(df.columns)} colonnes")
    return df

# ===========================
# CELLULE 4 : Définir les chemins S3
# ===========================
s3_paths = {
    "InfoClimat": "data_sync/InfoClimat_France/2026_01_12_1768220353795_0.jsonl",
    "WeatherIchtegem": "data_sync/weather_iichtegem/2026_01_12_1768224585495_0.jsonl",
    "WeatherLaMadeleine": "data_sync/WeatherUnderground_LaMadeleine/2026_01_12_1768226046168_0.jsonl"
}

# ===========================
# CELLULE 5 : Lecture des données
# ===========================
df_infoclimat = read_jsonl_from_s3(s3_paths["InfoClimat"])
df_ichtegem = read_jsonl_from_s3(s3_paths["WeatherIchtegem"])
df_lamadeleine = read_jsonl_from_s3(s3_paths["WeatherLaMadeleine"])

# ===========================
# CELLULE 6 : Connexion MongoDB Atlas
# ===========================
# ===========================
# Connexion MongoDB Atlas sécurisée
# ===========================

import os
from pymongo import MongoClient
import certifi
from urllib.parse import quote_plus  # pour encoder le mot de passe

# Charger les variables d'environnement (à avoir exporté dans le terminal avant le lancement de Jupyter)
MONGO_USER = os.getenv("MONGO_USER")
MONGO_PASSWORD = os.getenv("MONGO_PASSWORD")
MONGO_CLUSTER = os.getenv("MONGO_CLUSTER")
MONGO_DB = os.getenv("MONGO_DB")


# Construire l'URI de connexion
mongo_uri = f"mongodb+srv://{MONGO_USER}:{MONGO_PASSWORD}@{MONGO_CLUSTER}/{MONGO_DB}?retryWrites=true&w=majority"

# Connexion MongoDB Atlas
client = MongoClient(
    mongo_uri,
    tls=True,                  # TLS obligatoire pour Atlas
    tlsCAFile=certifi.where()  # Certificat SSL
)

# Sélection de la base
db = client[MONGO_DB]

# Test de connexion : lister les collections
try:
    collections = db.list_collection_names()
    print("Connexion MongoDB Atlas établie ✅")
    print("Collections disponibles :", collections)
except Exception as e:
    print("Erreur connexion MongoDB :", e)


# ===========================
# CELLULE 7 : Insertion des données
# ===========================
# Supprime les anciennes données si existantes
db["InfoClimat"].delete_many({})
db["WeatherIchtegem"].delete_many({})
db["WeatherLaMadeleine"].delete_many({})

# Insertions
db["InfoClimat"].insert_many(df_infoclimat.to_dict(orient="records"))
db["WeatherIchtegem"].insert_many(df_ichtegem.to_dict(orient="records"))
db["WeatherLaMadeleine"].insert_many(df_lamadeleine.to_dict(orient="records"))

print("✅ Données insérées dans MongoDB")

# ===========================
# CELLULE 8 : Mesure de la qualité des données
# ===========================
def measure_quality(df, collection_name):
    total = len(df)
    missing = df.isna().sum().sum()  # nombre total de valeurs manquantes
    quality = (1 - missing / (total * len(df.columns))) * 100
    print(f"Taux de qualité pour {collection_name} : {quality:.2f}%")

measure_quality(df_infoclimat, "InfoClimat")
measure_quality(df_ichtegem, "WeatherIchtegem")
measure_quality(df_lamadeleine, "WeatherLaMadeleine")


✅ Librairies importées
AWS_ACCESS_KEY_ID chargé : True
AWS_SECRET_ACCESS_KEY chargé : True
✅ Connexion S3 établie pour le bucket greencoop-forecast2-raw
Lecture data_sync/InfoClimat_France/2026_01_12_1768220353795_0.jsonl : 1 lignes, 6 colonnes
Lecture data_sync/weather_iichtegem/2026_01_12_1768224585495_0.jsonl : 1899 lignes, 12 colonnes
Lecture data_sync/WeatherUnderground_LaMadeleine/2026_01_12_1768226046168_0.jsonl : 1908 lignes, 12 colonnes
Connexion MongoDB Atlas établie ✅
Collections disponibles : ['WeatherLaMadeleine', 'test_migration', 'WeatherData', 'InfoClimat', 'WeatherIchtegem']
✅ Données insérées dans MongoDB
Taux de qualité pour InfoClimat : 100.00%
Taux de qualité pour WeatherIchtegem : 99.97%
Taux de qualité pour WeatherLaMadeleine : 99.75%

📌 Script complet. Pour le logigramme, formaliser :
Collecte S3 → Lecture fichiers → Connexion MongoDB → Insertion → Mesure qualité
